In [92]:
# Variational autoencoder in Numpy without AI or reference code
#
# Honest list of all resources used
#   Feed-forward networks: Michael Nielsen, Neural Networks and Deep Learning, Ch. 1 and Ch. 2, without reading reference code examples
#   Gradients and backpropagation: 
#       -> 3Blue1Brown: Backpropagation, intuitively / Backpropagation calculus
#       -> ritvikmath: "Backpropogation: Data Science Concepts"
#       -> https://tutorial.math.lamar.edu/classes/calci/linearapproximations.aspx
#       -> Softmax function: https://en.wikipedia.org/wiki/Softmax_function / https://medium.com/data-science/derivative-of-the-softmax-function-and-the-categorical-cross-entropy-loss-ffceefc081d1
#   SGD:
#       -> https://en.wikipedia.org/wiki/Stochastic_gradient_descent 
#       -> https://www.geeksforgeeks.org/deep-learning/adam-optimizer/
#       -> Jacob Murri (UCLA), Math 118 Lecture Notes 
#       -> 
#
#   Autoencoders:
#       -> Durk Kingma, Max Welling: An Introduction to Variational Autoencoders 
#               (I had to do a lot of reading to understand what they were saying, since I've never taken Bayesian stats)
#       -> https://apxml.com/courses/introduction-autoencoders-feature-learning/chapter-3-how-autoencoders-learn/autoencoder-loss-functions

In [93]:
import numpy as np
import copy

In [94]:
# =========  1: The forward pass ===========
# 1.1 Creating the linear layers 
# A linear layer's weight matrix has size (output, input)
# 
# Per-neuron outputs per scalar j are ∑_i (w_j)_i * x_i + b_j, where i is the input dimension, and j is the output dimension
# Hence the j'th row of W corresponds to the vector w_j 
# All rows dot the same vector x
# -> Output of len j is the composition W @ x + b

class Layer(): 
    def function(self):
        return NotImplementedError
    def grad(self):
        return NotImplementedError
    index = None
    output = None
    next = None #Pointers for next and prev
    prev = None

class LinearLayer(Layer): 
    weights: np.array #(output_size, input_size)
    bias = np.array #(output_size, )
    def __init__(self, input_size: int, output_size: int): 
        rng = np.random.default_rng()
        self.weights = rng.random((output_size, input_size)) #Initialize with random weights
        self.bias = rng.random(output_size)    
    def __call__(self, x: np.array): 
        return self.weights @ x + self.bias
    def size(self): 
        return f"In: {self.weights.shape[-1]}, out: {self.weights.shape[0]}"
    def __repr__(self):
        return(f"LinearLayer: {self.size()}")
    def grad(self, *args, **kwargs): 
        return self.weights.T

layer = LinearLayer(input_size=8, output_size=3)
input = np.array([1,1,1,1,1,1,1,1])
layer(input)

array([3.48429034, 3.88986724, 4.34452707])

In [95]:
# 1.2 Non-linear functions
# In a perceptron, we take a singular output w1x1 + w2x2 + ... wnxn, same as dot product w * x, 
# We "accept" if the sum exceeds some certain number, b, otherwise we reject
# Mathematically equivalent to H(w*x + b)
# Because of the piecewise behavior, over multiple layers a small weight change that modifies 0->1 can drastically change behavior
# sigmoid: maps R -> (0,1), smooth version of H(x)
# Because vectorized functions are R^n -> R^n, then the derivative with respect to an input vector will also have to technically be a diagonal matrix
# where all off-diagonal terms are 0

class Nonlinear(Layer):
    name = "nonlinear"
    def __call__(self, z: np.array): 
        return self.function(z)
    def __repr__(self):
        return f"{self.name}"

class Sigmoid(Nonlinear):
    def __init__(self): 
        self.function = lambda z: (1+np.exp(-z))**-1
        self.grad = lambda z: np.diag(self.function(z)*(1-self.function(z)))
        self.name = "sigmoid"
class ReLU(Nonlinear):
    def __init__(self):
        self.function = lambda z: np.maximum(0, z)
        self.grad = lambda z: np.diag(np.where(z > 0, 1,0))
        self.name = "relu"

    
# 1.21 final activation function thru softmax
# We want to turn the final linear layer output into a probability distribution
class Softmax(Nonlinear):
    def __init__(self): 
        self.name = "softmax"
        self.function = lambda z: np.exp(z) / np.sum(np.exp(z))
    # Since softmax is not a vectorized function, then each output depends on all inputs,
    # making the _functionient a jacobian matrix 
    # Although the jacobian matrix contains functions of x_i, the softmax derivative only uses 
    # simple transformations of the original output, since diagonal entries are (y_1)(1-y_1) and 
    # off-diagonal entries are -y_i*y_j, where y_i = exp(x_i).
    # The (i,j)'th entry is y_i {i=j} - y_i*y_j
    # => diag(y) - yy^T
    def grad(self, z):
        y = self.function(z)
        return (np.diag(y) - np.outer(y, y))
        

In [96]:
# 1.3 Creating a neural network
# Add linear or nonlinear modules with `add`, can be any function that works on numpy arrays
class NeuralNetwork: 
    first_module = None
    last_module = None
    input_dim, output_dim = None, None
    parameters = []
    def get_modules(self):
        current = self.first_module
        modules = [current]
        while current.next:
            current = current.next
            modules.append(current)
        return modules
    
    def add(self, module): 
        if self.first_module is None: 
            self.first_module = module
            self.last_module = module
            if isinstance(module, LinearLayer):
                self.input_dim = module.weights.shape[0]
            self.first_module.prev = self.input
            self.input.next = self.first_module
        else:
            module.prev = self.last_module
            self.last_module.next = module
            self.last_module = module
        # Add parameters and update output dimensionality if it's a linear layer
        if isinstance(module, LinearLayer): 
            self.parameters.append({"weights": module.weights, "biases": module.bias})
            self.output_dim = module.weights.shape[-1]
    def __init__(self): 
        self.modules = []
        self.parameters = []
        self.input = Layer()
    def forward(self, input):
        self.input.output = input
        output = input
        current_module = self.first_module
        while current_module:
            output = current_module(output)
            current_module.output = output
            current_module = current_module.next
        return output
    def __repr__(self):
        for i, module in enumerate(self.get_modules()):
            print(f"module {i}: {module}")
    def __add__(self, NN2): 
        # This method chains together the output of the neural network to NN
        # Returns a new NeuralNetwork, keeping the originals with their current modules
        # Updates to the new NN will affect the current modules
        NN = NeuralNetwork()
        for module in self.get_modules():
            NN.add(copy.copy(module))
        for module in NN2.get_modules():
            NN.add(copy.copy(module))
        NN.input_dim = self.input_dim
        NN.output_dim = NN2.output_dim or self.output_dim

        return NN


        

NN = NeuralNetwork()
NN.add(module=LinearLayer(input_size=10, output_size=8))
NN.add(module=Sigmoid())
NN.add(module=LinearLayer(input_size=8, output_size=6))
NN.add(module=Sigmoid())
NN.add(module=LinearLayer(input_size=6, output_size=2))
NN.add(module=Softmax())


In [97]:
# =========  2: Using gradients to update the weights in neural networks ===========
# 2.0 Loss is a function of the weights
# We initialize our neural network with random weights and biases for the linear layers. 
# If we put in an input, the output is probably wrong in most cases.   
#
#This makes it that if we can quantify how "wrong" the random-weight answer is, 
# then it is possible to find some other setting for all the weights and biases 
# inside the neural network that make the answer *less wrong* 
#
# In this case, a neural network is a function f(weights + biases) from the 
# parameter dimension to the output dimension, therefore the loss is a
# function Loss(weights + biases) -> R
# 
# Furthermore, the neural network itself should be a differentiable function by
# the chain rule from calc, so as long as our loss function is differentiable 
# we can find the partial derivative of Loss(w + b) with respect to every individual w, b
#
# By the first order linear approximation, f(a+∆x) ≈ f(a) + f'(a)∆x 
# Therefore if we move in direction of -f'(a) we reduce f (basic principle of GD)

# 2.1 MSE 
# Since our goal is a autoencoder, which does a recreation, MSE is a more natural metric than Cross-entropy loss
#
# When a neural network with randomized weights spits out an output distribution,
# we have to score the output distribution based on something. 
# MSE is a reliable metric, the squared term punishes large errors while allowing small ones to exist. 
# This makes our loss error prioritize correcting the biggest differences from training reference while
# caring less about being perfectly precise on nearly-good weight differences
# It's nice and differentiable with the derivative being a linear function


class LossFunction(): 
    def __call__(self, *args, **kwargs):
        return self.function(*args, **kwargs)
    def function(self, *args, **kwargs): 
        raise NotImplementedError
    def grad(self, *args, **kwargs):
        raise NotImplementedError

class MeanSquaredError(LossFunction):
    def function(self, x, ref): 
        return np.sum((x-ref)**2)/(2*len(x))
    def grad(self, x, ref): 
        return (x-ref)/len(x)

# 2.2 Recursive chain rule applications make it easy to find individual derivatives
# I had to write this in a markdown below for the math
# For a given output neuron, its activation depends on the weights connecting to it and 
# its bias, but also the activations that all the neurons prior to it connected 
# The later a given operation is, we can relate it to dC more easily,
# If we can anchor somewhere and find a recursive formula between layers given our repeated 
# LinearLayer + sigmoid structure, we can easily recreate the chain rule with less computation
# For instance, if we can find a relation between dCdz2, and dz2dz1, we can find dC with respect 
# To any linear layer output z. 
# 
# since z is a composition of w and b, then one more chain rule application can easily find dCdw or dCdb

$$C = \frac{1}{2n}\sum_k (\hat{y}_k - y_k)^2$$
$$\text{where the neural network output } \hat y = \text{softmax}(z^{(L)}) $$
$$ \frac{\partial C}{\partial z^{(L)}_j} = \sum_i \frac{\partial \hat y_i }{\partial z^{(L)}_j} \frac{\partial C}{\partial \hat y _i}$$
$$\sigma \text{ is a nice vectorized function, but all values } y_i \text{ depend on } z^{(L)}_j \text{ in softmax}$$
$$\frac{\partial \hat y_i}{\partial z_j} = \begin{cases} \hat y_i - \hat y_i \hat y_j, \quad i = j \\ - \hat y_i \hat y_j, \quad i \neq j \end{cases} \implies \frac{\partial \hat y}{\partial z} = \text{diag}( \hat y) - \hat y \hat y^\top \text{ (A jacobian)}$$
$$\boxed{ \frac{\partial C}{\partial z^{(L)}} = \frac{1}{n} \left((\text{diag}(\hat y) - \hat y \hat y^\top) (\hat y - y) \right)}$$

$$\frac{\partial C}{\partial z^{(L-1)}} = \frac{\partial x^{(L-1)}}{\partial z^{(L-1)}} \frac{\partial z^{(L)}}{\partial x^{(L-1)}} \frac{\partial C}{\partial z^{(L)}}$$
$$\text{break to calc derivatives...}$$

$$ x^{(L)}_i = \sigma(z^{(L)}_i) \rightarrow \frac{\partial x_i}{\partial z^{(L)}_j} = \begin{cases} \sigma(z^{(L)}_i)(1-\sigma(z^{(L)}_i)), \quad i = j \\ 0, \quad \quad i \neq j\end{cases}
$$
$$z^{(L)} = W^{(L)}x^{(L-1)} + b^{(L)} \implies \frac{\partial{z^{(L)}}}{\partial x^{(L-1)}} = (W^{(L)})^\top$$
$$ \boxed{\implies \frac{\partial C}{\partial z^{(L-1)}} = \text{diag}\left(x^{(L-1)} \odot (1-x^{(L-1)}) \right) (W^{(L)})^\top \frac{\partial C}{\partial z^{(L)}}}$$



$$\text{Recursion on weights:}$$
$$\frac{\partial C}{\partial w_{ij}} = \sum_k \frac{\partial z^{(L)}_k}{\partial w^{{(L)}}_{i,j}} \frac{\partial C}{\partial z^{(L)}_k}$$
$$z^{(L)}_k = \sum_j w_{k,j}x_j + b^{(L)}_k \implies \frac{\partial z^{(L)}_k}{\partial w^{(L)}_{i, j}} = \begin{cases} x_j^{(L)}, \quad i = k \\ 0,\quad \text{ otherwise}\end{cases}$$
$$\text{therefore } \frac{\partial C}{\partial w^{(L)}_{ij}} = \frac{\partial z_i^{(L)}}{\partial w_{i, j}^{(L)}} \frac{\partial C}{\partial z^{{(L)}}_i} = x_j \frac{\partial C}{\partial z^{(L)}_i}, \quad \left[\frac{\partial C}{\partial W^{(L)}} \right]_{ij} = \frac{\partial C}{\partial z^{(L)}_i} x^{(L-1)}_j$$
$$ \boxed{\implies \frac{\partial C}{\partial W^{(L)}} = \frac{\partial C}{\partial z^{(L)}} \otimes x^{(L-1)}}$$
$$\text{Recursion on biases:}$$
$$z^{(L)}_k = \sum_j w_{k,j}^{(L)}x_j^{(L-1)} + b^{(L)}_k \implies \frac{\partial z^{(L)}_k}{\partial b^{(L)}_{i}} = \begin{cases}1, \quad i = k \\ 0, \quad i \neq k \end{cases} $$
$$\implies \frac{\partial C}{\partial b^{(L)}_i} = \frac{\partial C}{\partial z^{(L)}_i}$$
$$\boxed{\frac{\partial C}{\partial b^{(L)}} = \frac{\partial C}{\partial z^{(L)}}}$$



In [98]:
# Starting at the last layer, compute the first gradients dC/dz^L using only the last layer
# Then apply recursively

# Uses original loss from initialization to recursively compute error with respect to layers
# Final nonlinear output z^L  ...
# No idea how pytorch implements this, so this may be very inefficient
# Loss functions propagate gradients straight backwards (since they are numpy-vectorized)
# Linear layers distribute gradients from earlier layers to affect later layers 
# 
        # While we could technically chain rule derivatives backwards, all our neural networks
        # are setup in a (LinearLayer(x^{L-1}) -> z^{L} -> o(z) -> x ) fashion
        # Hypothetically though, we could easily calculate derivatives by defining module.grad and 
        # chaining together module.grad(module.prev.output) / module.grad(module.output) for implicit cases
        # 
        # Or preferrably, storing gradients when the forward pass hits
        # 
        # For the final group, find dCdzL
        # For the lower groups, dCdzL_ with the recursion

class loss:
    output_loss = 1
    outputs = None
    targets = None
    def __init__(self, model, loss_fn, outputs, targets): 
        self.NN = model
        self.loss_fn = loss_fn
        self.outputs = outputs
        self.targets = targets
        self.output_loss = self.loss_fn(outputs, targets)
    def update(self, outputs, targets):
        self.output_loss = self.loss_fn(outputs, targets) 
        return self.output_loss

    def backward(self): 
        last = self.NN.last_module #Nonlinear module, something like Sigmoid or Softmax
        dCdz = last.grad(last.prev.output) @ self.loss_fn.grad(self.outputs, self.targets) #dydz * dLdy
        last.gr = dCdz
        self.dCdzs = [dCdz]
        self.xs = []
        # Walk backwards, using the chain rule to calculate derivatives
        # When you encounter a sigmoid layer o(z_) = x_, save dCdz_
        # Save the corresponding x^(L-1) for each dCdz^(L), shifted one unit back
        # This is all done using regular matmul
        current = last.prev
        while current.prev is not None: 
            dmdm_ = current.grad(current.prev.output)
            dCdm = current.next.gr
            current.gr =  dmdm_@dCdm
            if isinstance(current, Nonlinear):
                self.dCdzs.append(current.gr)
                self.xs.append(current.output)
            current = current.prev 
        self.xs.append(self.NN.input.output)

        # Find dCdW with the outer product
        self.dCdWs = [
            np.outer(dCdz, x) for dCdz, x in zip(self.dCdzs, self.xs)
        ]
        return [
            {"d_weights": dCdW, "d_biases": dCdb} 
            for dCdW, dCdb in zip(reversed(self.dCdWs), reversed(self.dCdzs))
        ]

In [99]:
class Optimizer:
    NN: NeuralNetwork
    lr: int
    stp: int=0
    def __init__(self, NN, lr): 
        self.NN = NN
        self.lr = lr

class SGD(Optimizer): 
    def __init__(self, NN, lr, momentum: float=0): 
        super().__init__(NN, lr)
        self.momentum = momentum
        self.m_prev = []
    def step(self, grad): #Stock pytorch probably puts grad into the model.parameters object, but I don't want to do that
        self.stp += 1
        linear_layers = []
        for module in self.NN.get_modules(): 
            if isinstance(module, LinearLayer):
                linear_layers.append(module)
        i = 0
        for L in linear_layers: 
            grad_W = grad[i]["d_weights"]
            grad_b = grad[i]["d_biases"]
            # on initialization, there is no m_prev, so replace m_ with zeroes
            m_W_prev = self.m_prev[i]["m_weights"] if i < len(self.m_prev) else np.zeros_like(grad_W)
            m_b_prev = self.m_prev[i]["m_biases"] if i < len(self.m_prev) else np.zeros_like(grad_b)
            # Step and update weights
            m_W = self.momentum*m_W_prev + (1-self.momentum)*grad_W
            m_b = self.momentum*m_b_prev + (1-self.momentum)*grad_b
            L.weights -= self.lr*m_W
            L.bias -= self.lr*m_b
            i += 1
            if self.momentum: #Save to same index for momentum 4 next training example
                if i < len(self.m_prev): 
                    self.m_prev[i] = {"m_weights": m_W, "m_biases": m_b} #Override
                else: 
                    self.m_prev.append({"m_weights": m_W, "m_biases": m_b}) #if it doesn't exist, just append it
            
class Adam(Optimizer):
    # With SGD, we are only approximating the first-order gradient, often badly
    # So what use would approximating the parameters^2 terms for the second derivative? 
    # Our sample gradients are from a distribution, and so if we can find the variance
    # I sort of couldn't find an explanation online for how the gradient variance works, 
    # But based on the formula, a basic idea is that:  
    #   Var(dC) = E(dC^2) - E(dC)^2 by basic stats 
    #   sqrt(E(dC^2)) ∝ E(dC) so a large gradient implies a large variance
    #   Large gradient means big steps are dangerous, so scale the learning rate proportionally by sqrt(E(dC^2))
    def __init__(self, NN, lr, b1: float, b2: float): 
        super().__init__(NN, lr)
        self.b1 = b1
        self.b2 = b2
        self.m_prev = []
        self.v_prev = []
    def step(self, grad): 
        self.stp += 1
        linear_layers = []
        for module in self.NN.get_modules(): 
            if isinstance(module, LinearLayer):
                linear_layers.append(module)
        i = 0
        for L in linear_layers:
            #First moment
            grad_W = grad[i]["d_weights"]
            grad_b = grad[i]["d_biases"]
            m_W_prev = self.m_prev[i]["m_weights"] if i < len(self.m_prev) else np.zeros_like(grad_W)
            m_b_prev = self.m_prev[i]["m_biases"] if i < len(self.m_prev) else np.zeros_like(grad_b)
            v_W_prev = self.v_prev[i]["v_weights"] if i < len(self.v_prev) else np.zeros_like(grad_W)
            v_b_prev = self.v_prev[i]["v_biases"] if i < len(self.v_prev) else np.zeros_like(grad_b)
            #Step
            m_W = self.b1*m_W_prev + (1-self.b1)*grad_W
            m_b = self.b1*m_b_prev + (1-self.b1)*grad_b
            v_W = self.b2*v_W_prev + (1-self.b2)*(grad_W**2)
            v_b = self.b2*v_b_prev + (1-self.b2)*(grad_b**2)

            #Actually, this is wrong, the exponent is the current training iteration, so replace it with a step counter that saves in Optimizer objects
            hat_m_W = m_W/(1-self.b1**(self.stp))
            hat_m_b = m_b/(1-self.b1**(self.stp))
            hat_v_W = v_W/(1-self.b2**(self.stp))
            hat_v_b = v_b/(1-self.b2**(self.stp))

            L.weights -= self.lr*(hat_m_W/(np.sqrt(hat_v_W)+1e-6))
            L.bias -= self.lr*(hat_m_b/(np.sqrt(hat_v_b)+1e-6))
            i += 1
            #Save at same index
            if i < len(self.m_prev): #Since they get populated at the same time, just do one length check
                self.m_prev[i] = {"m_weights": m_W, "m_biases": m_b} 
                self.v_prev[i] = {"v_weights": v_W, "v_biases": v_b} 
            else: 
                self.m_prev.append({"m_weights": m_W, "m_biases": m_b})
                self.v_prev.append({"v_weights": v_W, "v_biases": v_b})
            

In [100]:
input = np.array([1,1,1,1,1,1,1,1,1,1,])
output = NN.forward(input)
labels = np.array([0.6, 0.4])
print(f"Output: {output}")

MSE = MeanSquaredError()
l = loss(NN, MSE, output, labels)
print(f"Output loss: {l.output_loss}")

grad = l.backward()

Output: [0.21810326 0.78189674]
Output loss: 0.07292255889354449


In [101]:
NN_copy = copy.deepcopy(NN)
ad = Adam(NN, 0.01, 0.5, 0.5)
ad.step(grad)
sg = SGD(NN_copy, 0.1, 0.5)
sg.step(grad)

output2 = NN.forward(input)
output3 = NN_copy.forward(input)
print(f"Adamized output: {output2}, loss: {l.update(output2, labels)}")
print(f"SGD output: {output3}, loss: {l.update(output3, labels)}")

Adamized output: [0.24294952 0.75705048], loss: 0.06374252374437896
SGD output: [0.22574055 0.77425945], loss: 0.0700350690115519


In [102]:
# Variational Autoencoder
#
# Autoencoders themselves are pretty simple: minimize reconstruction error, and the model finds an approximate
# parameter distribution in its lower dimension for each category. To me, I like to think of autoencoders as a nonlinear version of PCA
# where the latent space is our low-rank projection V^Tx, with r kept components corresponding to the dimension r of the smallest network layer
# While it can find distributions here pretty well, an arbitrary combination in latent space is not guaranteed 
# to look like anything after reconstruction, so it's not good at generalization/generation
# 
# The trick with VAEs is to minimize latent loss in addition to reconstruction loss, meaning that it forces the lower-dimensional
# distribution to resemble whatever prior we think the latent space has 
# The paper says how any sufficiently commplicated function can map any arbitrary distribution to another arbitrary distribution, meaning that 
# the VAE has to learn that sufficiently complicated function that can turn a nice distribution (e.g the gaussian) into our output distribution
#
# At a very high level, we just penalize the approximated latent distribution q(z|x) with something like KL, 
# even though the Kingma I read discusses this in much more detail I can't really understand (ELBO and true posteriors and monte-carlo estimators, etc)
# The Encoder portion outputs a distribution, which is penalized to resemble our prior N(0,1)
# This distribution is sampled and fed to the decoder for recreation, and this is also penalized to create a good reconstruction
#
# Sampling from a distribution means that we also need to model the variance, which can be done with... 
# A secondary neural network alongside the encoder step for variance
# Standard implementations are said to use logvar for numerical stability, 
# which can easily be converted back with exp

class Encoder():
    mean_head = NeuralNetwork()
    logvar_head = NeuralNetwork()
    def add(self, module):
        self.mean_head.add(module)
        self.logvar_head.add(copy.deepcopy(module))

    def forward(self, input):
        mu = self.mean_head.forward(input)
        logvar = self.logvar_head.forward(input)
        return mu, logvar
    
encoder = Encoder()
encoder.add(LinearLayer(16, 8))
encoder.add(Sigmoid())
encoder.add(LinearLayer(8,4))
encoder.add(Sigmoid())

# Encoders start out randomly outputting the same mean and variance vectors, but since the weight matrix copies 
# are different, we can train them individually
print(f"encoder mean head weights are encoder logvar head: {encoder.mean_head.first_module.weights is encoder.logvar_head.first_module.weights }")

mu, logvar = encoder.forward(np.arange(0,16))
label = np.array([0,0,1,0])
mean_loss = loss(encoder.mean_head, MSE, mu, label)
print(f"output: {mu}, label: {label} ==> mean loss: {mean_loss.output_loss}")

mean_grad = mean_loss.backward()
mean_optim = Adam(encoder.mean_head, lr=0.1, b1=0.5, b2=0.5)
mean_optim.step(mean_grad)
new_mu, new_logvar = encoder.forward(np.arange(0,16))
print(f"new output: {new_mu} => mean loss: {mean_loss.update(new_mu, label)}")
print(f"logvar is the same? {new_logvar == logvar}")


encoder mean head weights are encoder logvar head: False
output: [0.98005442 0.99056186 0.98726819 0.99244758], label: [0 0 1 0] ==> mean loss: 0.3658542195071748
new output: [0.95233812 0.97711007 0.99466911 0.9816353 ] => mean loss: 0.3531660332155137
logvar is the same? [ True  True  True  True]


In [103]:
# Math break 2: 
# The VAE loss function we're using is C(y, ref, mu, logvar) = MSE(y, ref) + KL(mu, logvar)
# The KL term does not depend on any of the layers in the decoder, hence dKLdW for decoder
# weights is zero
# Encoder w&b can affect the KL term in the loss,
# All w&b can affect the reconstruction term

$$C(y, \mu, \ell) = \text{MSE}(\hat y(\mu, \ell)) + \beta \cdot KL(\mu, \ell)$$
$$KL(\hat \mu, \hat \ell) = -\frac{1}{2} \sum_d (1+ \hat \ell_d - \hat \mu^2_d - \exp(\hat \ell_d)), \quad \hat \ell = \log(\hat \sigma^2)$$

$$ \frac{\partial KL}{\partial z^{\hat \mu, (L)}} = \frac{\partial \hat \mu}{\partial z^{\hat \mu, (L)}} \frac{\partial KL}{\partial \hat \mu}=  \hat \mu(1-\hat \mu) \odot \mu \quad \text{since } \hat\mu = \sigma(z^{\hat \mu, (L)})$$
$$\text{therefore } \frac{\partial KL}{\partial W^{\mu, (L)}} = \frac{\partial KL}{\partial z^{\mu, (L)}} \otimes x^{\mu, (L-1)}$$

$$\frac{\partial KL}{\partial z^{\hat \ell, (L)}} = \frac{\partial \hat \ell}{\partial z^{\hat \ell, (L)}} \frac{\partial KL}{\partial \hat \ell} = 
\hat \ell(1-\hat \ell) \ \odot \ \frac{1}{2}(\exp\hat \ell - 1)
$$


In [104]:
# The derivative of the MSE with respect to the mu and logvar would have been straightforward if not for the 
# reparameterization term r. Happilly, it is differentiable now. 
# In the case of the weights for mu, the contribution term of the MSE is exactly the same as it is in 
# the normal case, since drdu = I, nice
# In the logvar case, we have a different scaling matrix, which is a little bit more complicated, but 
# also simply a scalar
#
# Based on the derivatives, it seems that the smaller logvar and the sampled z are, the gradients get 
# less magnitude. This makes sense, since (i have no idea why this makes sense. Small gradients mean the 
# logvar head is not trained, meaning it's good where it currently is)
# This term actually wants to force logvar to 0, but making logvar 0 would increase the KL penalty 

$$\frac{\partial MSE(\hat y)}{\partial z^{\hat \mu, (L)}} = 
\frac{\partial \mu}{\partial z^{\mu, (L)}} \frac{\partial r}{\partial \mu }
\frac{z^{(0)}}{\partial r}\frac{\partial MSE(\hat y)}{\partial z^{(0)}}

$$
$$r(\hat\mu, \hat\ell) = \mu + z*\exp(\frac 1 2\ell); \  z \sim N(0,1), \quad z^{(0)} = W^{(0)} r + b^{(0)}, \quad $$
$$\frac{\partial MSE}{\partial \hat z^{\hat\mu, (L)}} = \hat \mu (1-\mu) \odot \vec 1 \odot (W^{(0)})^\top \frac{\partial MSE}{\partial z^{{0}}}$$
$$\text{In the same manner, }$$
$$\frac{\partial r}{\partial \ell} = \frac 1 2  z \exp(\frac 12  \hat \ell)$$
$$\implies \frac{\partial MSE}{\partial z^{\hat \ell, (L)}} = \hat \ell (1 - \hat \ell) \odot \frac 1 2 z \exp(\frac 12  \hat \ell) \odot (W^{(0)})^\top \frac{\partial MSE}{\partial z^{(0)}}

In [105]:
# Because our total loss combines MSE/Reconstruction error and scaled KL/latent error, 
# then the total contribution is as follows: 
# MSE/Reconstruction error term: 
#   - decoder: no change from default backpropagation
#   - encoder: no change in the mean-generating network, multiply weights by 1/2log(1/2l) for the logvar
# KL term
#   - decoder: no change
#   - encoder: multiply all KL contributions by the scaling factor b
#
# Pseudoalgorithm reusing existing code: 
#   - Create two seperate NeuralNetwork objects linking the encoder's mean head and variance head,
#     backprop on this combined neural network will yield the correct contribution of gradients for 
#     all mean-generating encoder and decoder modules
#   - Save the deepest dCdz from the decoder, perform the additional matrix multiplication by drdl, 
#     and then do backpropagation as normal
#   - Find the beta-scaled contribution of the KL divergence term for the encoder modules 

In [106]:
class Reparameterize(Nonlinear):
        # Sampling from the distribution defined by mu, var is equivalent to some function f(mu, var) 
        # which takes a random sample from N(mu, var), which is not differentiable
        # Luckily for us, we can simulate a random sample since X ~ N(mu, var) = mu + z*std, where z~N(0,1)
        # For a fixed sample, this is differentiable: dxdmu = 1, dzdvar = z
        def function(self, input):
            print(input)
            mu, logvar = input
            rng = np.random.default_rng()
            self.eps = rng.standard_normal(size=mu.size)
            return mu + self.eps*np.exp(0.5*logvar)    
        def grad(self, mu, logvar):
            # Training for the mean mode (overrides, pass in logvar to train logvar)
            # Again, since the reparametrization is R^n -> R^n, its gradient is a diagonal matrix
            if mu: 
                return np.identity(mu.size)
            else:
                return np.diag(self.eps)
        def __init__(self): 
            self.name = "reparameterization trick" 

In [107]:
class VariationalAutoencoder():
    def reparameterize(self, mu, logvar): 
        rng = np.random.default_rng()
        self.eps = rng.standard_normal(size=mu.size)
        return mu + self.eps*np.exp(0.5*logvar)
    
    def __init__(self, encoder, decoder):
        self.encoder = encoder
        self.decoder = decoder
        self.input_dim = encoder.mean_head.input_dim
        self.output_dim = decoder.output_dim
        self.latent_dim = decoder.input_dim

    def forward(self, input): 
        self.input = input
        mu, logvar = self.encoder.forward(input)
        sample = self.reparameterize(mu, logvar)
        output = self.decoder.forward(sample)
        return output, mu, logvar

decoder = NeuralNetwork()
decoder.add(LinearLayer(4,8))
decoder.add(Sigmoid())
decoder.add(LinearLayer(8,16))
decoder.add(Sigmoid())


In [110]:
input = np.array([0,1,0]+[0]*13) #(0,1,0...)
VAE = VariationalAutoencoder(encoder, decoder)
y, mu, logvar = VAE.forward(input)
vae_loss = VAELoss(VAE, MSE, 
                   outputs=y, targets=input, 
                   beta = 0.5)

In [111]:
test_loss = vae_loss.backward2()
test_loss.backward()

TypeError: unsupported operand type(s) for *: 'float' and 'NoneType'

In [69]:
# The implementation's loss function takes in the reconstruction output, as well as the latent distribution parameters u and o2
# Because of the sampling from the 

class VAELoss(LossFunction):
    def KL_normal(self, mu, logvar):
        return 0.5*np.sum((1+logvar - mu**2 - np.exp(logvar))) 
    def __init__(self, VAE, rc_loss, outputs, targets, beta: int=1, ):
        self.outputs = outputs
        self.targets = targets
        self.VAE = VAE
        self.rc_loss = rc_loss
        self.beta = beta
    def function(self, x, ref, mu, logvar):
        recon_error = self.rc_loss(x, ref)
        latent_error = self.KL_normal(mu, logvar)
        print(recon_error, latent_error)
        return self.rc_loss(x, ref) + self.beta*self.KL_normal(mu, logvar)


    def backward2(self):
        # Like backwards in normal loss, but with two return modules
        # Updates for the complete reconstruction module according to normal recursive formula
        # Updates for the logvar part of the encoder have KL contribution multiplied by beta
        #
        # Terms dependent on MSE:
        mean_reconstructor = self.VAE.encoder.mean_head + self.VAE.decoder 
        mr_loss = loss(model=mean_reconstructor, loss_fn=self.rc_loss, 
                       outputs=self.outputs, targets=self.targets)
        return mr_loss
        dRCdmr = mr_loss.backward()

        d_index = len([l for l in self.VAE.encoder.mean_head.get_modules() if isinstance(l, LinearLayer)])
        dRCdzs = [mr_loss.dCdzs[d_index]] # Get earliest dRCdz (last in the array, since we backpropagated)
        print(dRCdzs.shape)
        
    def backward(self): 
        # Like backwards in normal loss, but with two return modules
        # Updates for the complete reconstruction module according to normal recursive formula
        # Updates for the logvar part of the encoder have KL contribution multiplied by beta
        #
        # Terms dependent on MSE: 
        decoder_loss = loss(model=self.VAE.decoder, loss_fn=self.rc_loss, 
                            outputs = self.outputs, targets=self.targets)
        dRCdW_decoder = decoder_loss.backward()
        dRCdzs = [decoder_loss.dCdzs[-1]] # Get earliest dRCdz (last in the array, since we backpropagated)
        mean_module, logvar_module = self.VAE.encoder.mean_head.last_module, self.VAE.encoder.logvar_head.last_module

        # Calculate bridge between encoder/decoder layers through r(u, o^2) using the scaling factor drdl
        drdl = 0.5*self.VAE.eps*np.exp(0.5*logvar_module.output) #scaling factor for logvar, 0.5z*exp(0.5l)
        mu, logvar = mean_module.output, logvar_module.output 
        W_0 = self.VAE.decoder.first_module.weights #first decoder module 
        print(decoder_loss.dCdzs)

        dRCdz_logvar =  logvar*(1-logvar) *drdl * ( W_0.T @ dRCdzs[-1])
        dRCdz_mu = mu*(1-mu) * (W_0.T @ dRCdzs[-1]) #drdl = I

        xs_mu, xs_l = [], []
        dRCdzs_mu = [dRCdz_mu]
        dRCdzs_l = [dRCdz_logvar]

        while mean_module.prev is not None: #Parallel encoder modules so just iterate over one
            mean_module, logvar_module = mean_module.prev, logvar_module.prev
            if isinstance(mean_module, Sigmoid):
                print("sigmoid")
                x_mu, x_l = mean_module.output, logvar_module.output
                xs_mu.append(x_mu)
                xs_l.append(x_l)
                W_m, W_l = mean_module.next.weights, logvar_module.next.weights
                dRCdz_mu_ = x_mu*(1-x_mu) * (W_m.T @ dRCdzs_mu[-1])
                print(np.outer(dRCdz_mu, x_mu))
                dRCdz_l_ =  (W_l.T @ dRCdzs_mu[-1])
                dRCdzs_mu.append(dRCdz_mu_)
                dRCdzs_l.append(dRCdz_l_)
        xs_mu.append(self.VAE.input)
        xs_l.append(self.VAE.input)


        self.dRCdWs_mu = [ np.outer(dRCdz, x) for dRCdz, x in zip(dRCdzs_mu, xs_mu)]
        self.dRCdWs_l = [ np.outer(dRCdz, x) for dRCdz, x in zip(dRCdzs_l, xs_l)]
        return self.dRCdWs_mu
        # Terms dependent on KL:

    
        

